In [11]:
from core.lca import get_inventory_dataset, run_lca, compute_midpoint_contributions

In [12]:
# Brightway imports
import bw2data as bd
import brightway2 as bw

In [13]:
import pandas as pd

In [ ]:
BW_PROJECT = 'metallican' # insert your project name here
bd.projects.set_current(BW_PROJECT)
list(bd.databases)

# Extract production and demand data

In [14]:
demand_df = pd.read_excel(r'data/data_lci.xlsx', sheet_name='DEMAND')
lci_demand = pd.read_excel(r'data/data_lci.xlsx', sheet_name='lci_demand')

In [15]:
df_demand_agg = (demand_df
    .groupby(["Scenario", "Year", "Metal"])["Value_t"]
    .sum()
    .reset_index()
)


In [16]:
df_demand_cm = df_demand_agg[df_demand_agg['Scenario'] == 'Current Measures']
df_demand_nz = df_demand_agg[df_demand_agg['Scenario'] == 'Canada Net-zero']

In [17]:
df_demand_cm

,Scenario,Year,Metal,Value_t
33,Current Measures,2020,Cobalt,1.558924e+03
34,Current Measures,2020,Copper,1.720507e+06
35,Current Measures,2020,Nickel,6.273683e+05
36,Current Measures,2025,Cobalt,1.715008e+04
37,Current Measures,2025,Copper,1.947374e+06
38,Current Measures,2025,Graphite,1.104294e+02
39,Current Measures,2025,Lithium,4.734106e+03
40,Current Measures,2025,Nickel,6.605654e+05
41,Current Measures,2030,Cobalt,5.119824e+04
42,Current Measures,2030,Copper,2.298394e+06


# Set up BW and LCIs

# Associate LCI

In [23]:
from bw2data import Database

def build_demand_lci_mapping(df_map):
    """
    df_map columns:
        Metal, Activity, Reference Product, Location, Year, Scenario, DB_to_map
    """

    mapping = {}

    for i, row in df_map.iterrows():

        key = (
            row["Scenario"],
            str(row["Year"]),
            row["Metal"]
        )

        db_name = row["DB_to_map"]
        act_name = row["Activity"]
        ref_prod = row["Reference Product"]
        loc = row["Location"]

        try:
            candidates = Database(db_name).search(act_name, limit=500)

            filtered = [
                a for a in candidates
                if a["name"] == act_name
                and a["reference product"] == ref_prod
                and a["location"] == loc
            ]

            mapping[key] = filtered[0] if filtered else None

        except:
            mapping[key] = None

    return mapping


In [24]:
lci_map = build_demand_lci_mapping(lci_demand)

In [25]:
lci_map

{('Current Measures', '2020', 'Cobalt'): None,
 ('Canada Net-zero',
  '2020',
  'Cobalt'): 'market for cobalt sulfate' (kilogram, World, None),
 ('Current Measures', '2025', 'Cobalt'): None,
 ('Canada Net-zero',
  '2025',
  'Cobalt'): 'market for cobalt sulfate' (kilogram, World, None),
 ('Current Measures', '2030', 'Cobalt'): None,
 ('Canada Net-zero',
  '2030',
  'Cobalt'): 'market for cobalt sulfate' (kilogram, World, None),
 ('Current Measures', '2035', 'Cobalt'): None,
 ('Canada Net-zero',
  '2035',
  'Cobalt'): 'market for cobalt sulfate' (kilogram, World, None),
 ('Current Measures', '2040', 'Cobalt'): None,
 ('Canada Net-zero',
  '2040',
  'Cobalt'): 'market for cobalt sulfate' (kilogram, World, None),
 ('Current Measures', '2020', 'Copper'): None,
 ('Canada Net-zero',
  '2020',
  'Copper'): 'market for copper, cathode' (kilogram, World, None),
 ('Current Measures', '2025', 'Copper'): None,
 ('Canada Net-zero',
  '2025',
  'Copper'): 'market for copper, cathode' (kilogram, Worl

# Calculate LCA for demand scenarios

In [ ]:
from bw2calc import LCA
import numpy as np

def calculate_demand_lca(df_demand_agg, lci_map, lcia_methods):
    """
    df_demand_agg columns:
        Scenario, Year, Metal, Value_t

    lci_map:
        dict[(Scenario, Year, Metal)] -> BW Activity
    """

    results = []

    for i, row in df_demand_agg.iterrows():

        scenario = row["Scenario"]
        year = str(row["Year"])
        metal = row["Metal"]
        qty = row["Value_t"]

        key = (scenario, year, metal)
        activity = lci_map.get(key, None)

        impact_results = {}

        if activity:
            try:
                lca = LCA({activity: qty}, lcia_methods[0])
                lca.lci()
                for method in lcia_methods:
                    lca.switch_method(method)
                    lca.lcia()
                    impact_results[method[2]] = lca.score
            except:
                impact_results = {m[2]: np.nan for m in lcia_methods}

        else:
            impact_results = {m[2]: np.nan for m in lcia_methods}

        results.append({
            "Scenario": scenario,
            "Year": year,
            "Metal": metal,
            "Demand_t": qty,
            **impact_results
        })

    return pd.DataFrame(results)
